In [0]:
CATALOG = "adwm_wh"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dimcustomer"

source_tables = [
    f"{CATALOG}.{SILVER_SCHEMA}.customer",
    f"{CATALOG}.{SILVER_SCHEMA}.person",
    f"{CATALOG}.{SILVER_SCHEMA}.businessentity",
    f"{CATALOG}.{SILVER_SCHEMA}.emailaddress",
    f"{CATALOG}.{SILVER_SCHEMA}.personphone",
    f"{CATALOG}.{SILVER_SCHEMA}.businessentityaddress",
    f"{CATALOG}.{SILVER_SCHEMA}.address",
    f"{CATALOG}.{SILVER_SCHEMA}.stateprovince",
    f"{CATALOG}.{SILVER_SCHEMA}.countryregion",
    f"{CATALOG}.{SILVER_SCHEMA}.addresstype"
]

print(f"Target table: {TARGET_TABLE}")
print("Source tables:")
for table_name in source_tables:
    print(f"  - {table_name}")

In [0]:
%sql
WITH source_prepared AS (
    SELECT
        cust.CustomerID AS CustomerAlternateKey,
        cust.AccountNumber,
        per.FirstName,
        per.MiddleName,
        per.LastName,
        trim(concat_ws(' ', per.Title, per.FirstName, per.MiddleName, per.LastName, per.Suffix)) AS FullName,
        per.Title,
        per.Suffix,
        per.PersonType,
        eml.EmailAddress,
        phn.PhoneNumber,
        addr.AddressLine1,
        addr.AddressLine2,
        addr.City,
        addr.StateProvinceID,
        st.Name AS StateProvinceName,
        addr.PostalCode,
        st.CountryRegionCode,
        ctry.Name AS CountryRegionName,
        addr_type.Name AS AddressType,
        cust.TerritoryID,
        per.EmailPromotion,
        ent.rowguid AS RowGUID
    FROM adwm_wh.silver.customer cust
    INNER JOIN adwm_wh.silver.person per
        ON cust.PersonID = per.BusinessEntityID
    INNER JOIN adwm_wh.silver.businessentity ent
        ON per.BusinessEntityID = ent.BusinessEntityID
    LEFT JOIN adwm_wh.silver.emailaddress eml
        ON per.BusinessEntityID = eml.BusinessEntityID
    LEFT JOIN adwm_wh.silver.personphone phn
        ON per.BusinessEntityID = phn.BusinessEntityID
    LEFT JOIN adwm_wh.silver.businessentityaddress ent_addr
        ON per.BusinessEntityID = ent_addr.BusinessEntityID
    LEFT JOIN adwm_wh.silver.address addr
        ON ent_addr.AddressID = addr.AddressID
    LEFT JOIN adwm_wh.silver.stateprovince st
        ON addr.StateProvinceID = st.StateProvinceID
    LEFT JOIN adwm_wh.silver.countryregion ctry
        ON st.CountryRegionCode = ctry.CountryRegionCode
    LEFT JOIN adwm_wh.silver.addresstype addr_type
        ON ent_addr.AddressTypeID = addr_type.AddressTypeID
),
source_data AS (
    SELECT
        CustomerAlternateKey,
        customer_record.AccountNumber AS AccountNumber,
        customer_record.FirstName AS FirstName,
        customer_record.MiddleName AS MiddleName,
        customer_record.LastName AS LastName,
        customer_record.FullName AS FullName,
        customer_record.Title AS Title,
        customer_record.Suffix AS Suffix,
        customer_record.PersonType AS PersonType,
        customer_record.EmailAddress AS EmailAddress,
        customer_record.PhoneNumber AS PhoneNumber,
        customer_record.AddressLine1 AS AddressLine1,
        customer_record.AddressLine2 AS AddressLine2,
        customer_record.City AS City,
        customer_record.StateProvinceID AS StateProvinceID,
        customer_record.StateProvinceName AS StateProvinceName,
        customer_record.PostalCode AS PostalCode,
        customer_record.CountryRegionCode AS CountryRegionCode,
        customer_record.CountryRegionName AS CountryRegionName,
        customer_record.AddressType AS AddressType,
        customer_record.TerritoryID AS TerritoryID,
        customer_record.EmailPromotion AS EmailPromotion,
        customer_record.RowGUID AS RowGUID,
        current_timestamp() AS ModifiedDate,
        sha2(
            concat_ws(
                '||',
                coalesce(cast(CustomerAlternateKey AS STRING), '∅'),
                coalesce(cast(customer_record.AccountNumber AS STRING), '∅'),
                coalesce(cast(customer_record.FirstName AS STRING), '∅'),
                coalesce(cast(customer_record.MiddleName AS STRING), '∅'),
                coalesce(cast(customer_record.LastName AS STRING), '∅'),
                coalesce(cast(customer_record.FullName AS STRING), '∅'),
                coalesce(cast(customer_record.Title AS STRING), '∅'),
                coalesce(cast(customer_record.Suffix AS STRING), '∅'),
                coalesce(cast(customer_record.PersonType AS STRING), '∅'),
                coalesce(cast(customer_record.EmailAddress AS STRING), '∅'),
                coalesce(cast(customer_record.PhoneNumber AS STRING), '∅'),
                coalesce(cast(customer_record.AddressLine1 AS STRING), '∅'),
                coalesce(cast(customer_record.AddressLine2 AS STRING), '∅'),
                coalesce(cast(customer_record.City AS STRING), '∅'),
                coalesce(cast(customer_record.StateProvinceID AS STRING), '∅'),
                coalesce(cast(customer_record.StateProvinceName AS STRING), '∅'),
                coalesce(cast(customer_record.PostalCode AS STRING), '∅'),
                coalesce(cast(customer_record.CountryRegionCode AS STRING), '∅'),
                coalesce(cast(customer_record.CountryRegionName AS STRING), '∅'),
                coalesce(cast(customer_record.AddressType AS STRING), '∅'),
                coalesce(cast(customer_record.TerritoryID AS STRING), '∅'),
                coalesce(cast(customer_record.EmailPromotion AS STRING), '∅'),
                coalesce(cast(customer_record.RowGUID AS STRING), '∅')
            ),
            256
        ) AS ChangeHash
    FROM (
        SELECT
            CustomerAlternateKey,
            max_by(
                named_struct(
                    'AccountNumber', AccountNumber,
                    'FirstName', FirstName,
                    'MiddleName', MiddleName,
                    'LastName', LastName,
                    'FullName', FullName,
                    'Title', Title,
                    'Suffix', Suffix,
                    'PersonType', PersonType,
                    'EmailAddress', EmailAddress,
                    'PhoneNumber', PhoneNumber,
                    'AddressLine1', AddressLine1,
                    'AddressLine2', AddressLine2,
                    'City', City,
                    'StateProvinceID', StateProvinceID,
                    'StateProvinceName', StateProvinceName,
                    'PostalCode', PostalCode,
                    'CountryRegionCode', CountryRegionCode,
                    'CountryRegionName', CountryRegionName,
                    'AddressType', AddressType,
                    'TerritoryID', TerritoryID,
                    'EmailPromotion', EmailPromotion,
                    'RowGUID', RowGUID
                ),
                named_struct(
                    'completeness_score',
                        CASE WHEN EmailAddress IS NOT NULL AND EmailAddress <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN PhoneNumber IS NOT NULL AND PhoneNumber <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN AddressLine1 IS NOT NULL AND AddressLine1 <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN City IS NOT NULL AND City <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN StateProvinceName IS NOT NULL AND StateProvinceName <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN PostalCode IS NOT NULL AND PostalCode <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN CountryRegionName IS NOT NULL AND CountryRegionName <> 'UNKNOWN' THEN 1 ELSE 0 END +
                        CASE WHEN AddressType IS NOT NULL AND AddressType <> 'UNKNOWN' THEN 1 ELSE 0 END,
                    'address_value', coalesce(AddressLine1, ''),
                    'email_value', coalesce(EmailAddress, ''),
                    'phone_value', coalesce(PhoneNumber, ''),
                    'rowguid_value', coalesce(RowGUID, '')
                )
            ) AS customer_record
        FROM source_prepared
        GROUP BY CustomerAlternateKey
    )
)
MERGE INTO adwm_wh.gold.dimcustomer AS target
USING source_data AS source
ON target.CustomerAlternateKey = source.CustomerAlternateKey
WHEN MATCHED AND source.ChangeHash <> sha2(
    concat_ws(
        '||',
        coalesce(cast(target.CustomerAlternateKey AS STRING), '∅'),
        coalesce(cast(target.AccountNumber AS STRING), '∅'),
        coalesce(cast(target.FirstName AS STRING), '∅'),
        coalesce(cast(target.MiddleName AS STRING), '∅'),
        coalesce(cast(target.LastName AS STRING), '∅'),
        coalesce(cast(target.FullName AS STRING), '∅'),
        coalesce(cast(target.Title AS STRING), '∅'),
        coalesce(cast(target.Suffix AS STRING), '∅'),
        coalesce(cast(target.PersonType AS STRING), '∅'),
        coalesce(cast(target.EmailAddress AS STRING), '∅'),
        coalesce(cast(target.PhoneNumber AS STRING), '∅'),
        coalesce(cast(target.AddressLine1 AS STRING), '∅'),
        coalesce(cast(target.AddressLine2 AS STRING), '∅'),
        coalesce(cast(target.City AS STRING), '∅'),
        coalesce(cast(target.StateProvinceID AS STRING), '∅'),
        coalesce(cast(target.StateProvinceName AS STRING), '∅'),
        coalesce(cast(target.PostalCode AS STRING), '∅'),
        coalesce(cast(target.CountryRegionCode AS STRING), '∅'),
        coalesce(cast(target.CountryRegionName AS STRING), '∅'),
        coalesce(cast(target.AddressType AS STRING), '∅'),
        coalesce(cast(target.TerritoryID AS STRING), '∅'),
        coalesce(cast(target.EmailPromotion AS STRING), '∅'),
        coalesce(cast(target.RowGUID AS STRING), '∅')
    ),
    256
) THEN UPDATE SET
    target.CustomerAlternateKey = source.CustomerAlternateKey,
    target.AccountNumber = source.AccountNumber,
    target.FirstName = source.FirstName,
    target.MiddleName = source.MiddleName,
    target.LastName = source.LastName,
    target.FullName = source.FullName,
    target.Title = source.Title,
    target.Suffix = source.Suffix,
    target.PersonType = source.PersonType,
    target.EmailAddress = source.EmailAddress,
    target.PhoneNumber = source.PhoneNumber,
    target.AddressLine1 = source.AddressLine1,
    target.AddressLine2 = source.AddressLine2,
    target.City = source.City,
    target.StateProvinceID = source.StateProvinceID,
    target.StateProvinceName = source.StateProvinceName,
    target.PostalCode = source.PostalCode,
    target.CountryRegionCode = source.CountryRegionCode,
    target.CountryRegionName = source.CountryRegionName,
    target.AddressType = source.AddressType,
    target.TerritoryID = source.TerritoryID,
    target.EmailPromotion = source.EmailPromotion,
    target.RowGUID = source.RowGUID,
    target.ModifiedDate = source.ModifiedDate
WHEN NOT MATCHED THEN INSERT (
    CustomerAlternateKey,
    AccountNumber,
    FirstName,
    MiddleName,
    LastName,
    FullName,
    Title,
    Suffix,
    PersonType,
    EmailAddress,
    PhoneNumber,
    AddressLine1,
    AddressLine2,
    City,
    StateProvinceID,
    StateProvinceName,
    PostalCode,
    CountryRegionCode,
    CountryRegionName,
    AddressType,
    TerritoryID,
    EmailPromotion,
    RowGUID,
    ModifiedDate
) VALUES (
    source.CustomerAlternateKey,
    source.AccountNumber,
    source.FirstName,
    source.MiddleName,
    source.LastName,
    source.FullName,
    source.Title,
    source.Suffix,
    source.PersonType,
    source.EmailAddress,
    source.PhoneNumber,
    source.AddressLine1,
    source.AddressLine2,
    source.City,
    source.StateProvinceID,
    source.StateProvinceName,
    source.PostalCode,
    source.CountryRegionCode,
    source.CountryRegionName,
    source.AddressType,
    source.TerritoryID,
    source.EmailPromotion,
    source.RowGUID,
    source.ModifiedDate
);

In [0]:
%sql
SELECT
    COUNT(*) AS dimcustomer_count,
    COUNT(DISTINCT CustomerAlternateKey) AS distinct_customeralternatekeys
FROM adwm_wh.gold.dimcustomer;

In [0]:
%sql
SELECT COUNT(*) AS dimcustomer_count
FROM adwm_wh.gold.dimcustomer;

In [0]:
%sql
SELECT *
FROM adwm_wh.gold.dimcustomer
ORDER BY CustomerAlternateKey
LIMIT 5;